# RoomBeacon Rental Data — Exploratory Data Analysis

Hệ thống **RoomBeacon** thu thập và tổng hợp dữ liệu tin đăng cho thuê phòng trọ, căn hộ mini và nhà trọ từ nhiều nền tảng trực tuyến khác nhau.

Trong vòng đời Khoa học Dữ liệu (Data Science Lifecycle) của RoomBeacon, quy trình phân tích dữ liệu khám phá được phân định rạch ròi thành hai giai đoạn độc lập:
1. **Khám phá Chất lượng Dữ liệu ban đầu (Initial / Data Quality EDA)**: Khám phá trực tiếp trên tập dữ liệu thô (**Bronze / Raw Data**) nhằm quan sát dữ liệu bẩn, phát hiện các trường bị thiếu (missing values), các định dạng thô bất thường (malformed values), lỗi trích xuất (parsing failures) và các giá trị dị biệt thô (raw outliers) để từ đó thiết lập **Quy tắc Làm sạch và Chuẩn hóa (Cleaning & Standardization Rules)**.
2. **Phân tích Khám phá Chuẩn hóa (Clean Analytical EDA)**: Thực hiện trên tập dữ liệu tầng **Silver** đã được làm sạch và ép kiểu chuẩn mực để khám phá các quy luật kinh tế thị trường cho thuê (giá thuê, diện tích, phân bố khu vực, tương quan đặc trưng) phục vụ cho kỹ thuật tạo đặc trưng (**Feature Engineering**) và tầng **Gold**.

# Chương 1. Giới thiệu

## 1.1 Bối cảnh dữ liệu và Vòng đời Khoa học Dữ liệu

Quy trình xử lý và luân chuyển dữ liệu chính thức trong hệ thống RoomBeacon tuân thủ vòng đời Khoa học Dữ liệu hiện đại:

$$\text{Source Websites} \longrightarrow \text{Crawler} \longrightarrow \text{Bronze JSON / MySQL Bronze} \longrightarrow \text{DuckDB Query / Flatten} \longrightarrow \text{Raw EDA Dataset} \longrightarrow \text{Pandas} \longrightarrow \text{Initial EDA} \longrightarrow \text{Cleaning Rules} \longrightarrow \text{Silver Parquet} \longrightarrow \text{Clean EDA} \longrightarrow \text{Feature Engineering} \longrightarrow \text{Gold} \longrightarrow \text{ML / Dashboards}$$

Tầng Silver không phải là điểm khởi đầu cho việc khám phá và phát hiện dữ liệu bẩn. Tầng Silver chỉ được xuất bản sau khi giai đoạn Initial EDA đã hiểu rõ các khiếm khuyết của dữ liệu thô và áp dụng thành công các quy tắc làm sạch.

## 1.2 Phân định vai trò các tầng dữ liệu

Hệ thống phân định rạch ròi trách nhiệm của từng thành phần kỹ thuật:

- **Bronze (Raw Ingestion)**: Tầng thu thập và lưu trữ toàn bộ các quan sát lịch sử thô (Append-only), bảo toàn nguyên vẹn 100% các giá trị gốc từ website (`price_raw`, `area_raw`, `location_raw`, `posted_at_raw`, `deposit_raw`, v.v.). Bronze bao gồm kho tệp tin thô **Bronze JSON** và cơ sở dữ liệu quan hệ **MySQL Bronze**, đóng vai trò làm bằng chứng gốc (audit trail), phục hồi thảm họa và cung cấp dữ liệu cho Initial EDA.
- **DuckDB (Analytical Query Engine)**: Công cụ truy vấn và biến đổi phân tích (OLAP Engine). DuckDB **không phải là một tầng dữ liệu** (như Bronze, Silver hay Gold), mà là công cụ tính toán giúp kết nối các bảng quan hệ MySQL Bronze, phẳng hóa cấu trúc quan hệ thành tập dữ liệu phân tích, đọc tệp Parquet và phục vụ SQL phân tích tương tác.
- **Raw EDA Dataset**: Tập dữ liệu phân tích phẳng hóa từ Bronze, bảo toàn đồng thời cả trường thô gốc (`*_raw`) và trường chuẩn hóa tạm thời (`*_value`) nhằm phục vụ cho việc đối chiếu và đánh giá chất lượng bộ trích xuất (Parser Profiling).
- **Silver (Cleaned & Standardized Layer)**: Tầng dữ liệu logic đã được làm sạch, chuẩn hóa và kiểm tra tính toàn vẹn cấu trúc danh tính (One-Row-Per-Listing) sau khi đã áp dụng các quy tắc chất lượng dữ liệu từ Initial EDA. Snapshot tầng Silver thường được lưu trữ vật lý dưới định dạng tệp tin tối ưu **Parquet** (lưu ý: *Parquet là định dạng tệp cột vật lý, Silver là tầng dữ liệu logic*).
- **Gold (Serving & Feature Products)**: Tầng sản phẩm dữ liệu cấp cao được tạo ra sau giai đoạn Clean EDA và Feature Engineering (chứa các đặc trưng tính toán như `price_per_m2`, `district_median_price`, `listing_age_days`, v.v.), sẵn sàng phục vụ cho các mô hình học máy (Machine Learning), khai phá dữ liệu (Data Mining) và bảng điều khiển trực quan (BI Dashboards).

## 1.3 Hai giai đoạn Phân tích Dữ liệu Khám phá (Two EDA Phases)

Việc phân tách rõ ràng hai giai đoạn EDA là nguyên tắc then chốt trong kỹ nghệ dữ liệu:

### Giai đoạn A: Khám phá Chất lượng Dữ liệu Thô (Initial / Data Quality EDA)
- **Nguồn dữ liệu**: Tập dữ liệu thô phẳng hóa từ Bronze (`Raw EDA Dataset`).
- **Mục tiêu**: Khám phá và hiểu rõ các khiếm khuyết của dữ liệu bẩn thông qua việc đối chiếu giữa giá trị thô gốc và giá trị parser trích xuất:
  - Tỷ lệ giá trị khuyết thiếu (Missing values) theo từng nguồn.
  - Các định dạng chuỗi bất thường (Malformed raw strings, đơn vị tỷ/triệu, m2/hecta).
  - Lỗi trích xuất của parser (Parsing failures, regex sai sót).
  - Dị biệt thô (Raw outliers) và hiện tượng quan sát lặp lại (Repeated observations).
  - Thiên lệch cấu trúc giữa các nguồn dữ liệu (Source bias).
- **Đầu ra**: Xây dựng bộ quy tắc làm sạch và chuẩn hóa (Cleaning & Standardization Rules).

### Giai đoạn B: Phân tích Khám phá Chuẩn hóa (Clean Analytical EDA)
- **Nguồn dữ liệu**: Snapshot tầng Silver (`rental_latest.parquet`).
- **Mục tiêu**: Phân tích các quy luật kinh tế và phân bố của thị trường cho thuê trên dữ liệu đã sạch:
  - Phân bố giá thuê phòng chuẩn (`price_vnd`) và diện tích chuẩn (`area_m2`).
  - Tương quan giữa giá thuê, diện tích và vị trí địa lý (Quận/Huyện).
  - Thống kê thị trường cho thuê tổng thể.
- **Đầu ra**: Đề xuất các đặc trưng và cấu trúc dữ liệu cho tầng Gold.

## 1.4 Mục tiêu phân tích và Định hướng tiếp theo

Mục tiêu tối thượng của chuỗi bài phân tích này là xây dựng bức tranh toàn cảnh về chất lượng dữ liệu và thị trường phòng trọ cho thuê.

> [!IMPORTANT]
> **BƯỚC TIẾP THEO (CURRENT NEXT STEP)**:
> Trước khi tiến hành phân tích khám phá chuyên sâu ở Chương 3, hệ thống cần chuẩn bị một tập dữ liệu **`Raw/Bronze EDA Dataset`** phẳng hóa bảo toàn đầy đủ các trường thô gốc (`price_raw`, `area_raw`, `location_raw`, `posted_at_raw`, `deposit_raw`) để phục vụ trọn vẹn cho giai đoạn **Initial / Data Quality EDA**.

Trong phần khởi tạo hiện tại ở Chương 2, notebook tiến hành kiểm tra môi trường làm việc và nạp thử nghiệm snapshot dữ liệu mẫu tầng Silver đã xuất bản (`data/silver/rental_latest.parquet`) để kiểm tra tính sẵn sàng của công cụ Pandas.

# Chương 2. Chuẩn bị môi trường và tải dữ liệu

## 2.1 Import các thư viện cần thiết

Khởi tạo các thư viện tiêu chuẩn phục vụ cho việc nạp dữ liệu và phân tích:
- `pandas`: Thao tác với cấu trúc bảng (DataFrame), nạp dữ liệu và thống kê mô tả.
- `numpy`: Các phép toán ma trận, vector hóa và xử lý giá trị khuyết thiếu.
- `matplotlib.pyplot`: Thư viện trực quan hóa và vẽ biểu đồ.
- `pathlib.Path`: Xử lý đường dẫn tập tin độc lập với nền tảng hệ điều hành.

In [1]:
import sys
from pathlib import Path
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

## 2.2 Cấu hình hiển thị Pandas

Thiết lập các tùy chọn hiển thị của Pandas nhằm nâng cao tính trực quan khi quan sát dữ liệu dạng bảng:
- Hiển thị đầy đủ tất cả các cột của DataFrame (`display.max_columns = None`).
- Giới hạn hiển thị tối đa 100 dòng (`display.max_rows = 100`).
- Định dạng hiển thị số thực với dấu phẩy phân cách hàng nghìn và 2 chữ số thập phân (`float_format`).

In [2]:
pd.set_option("display.max_columns", None)
pd.set_option("display.max_rows", 100)
pd.set_option("display.float_format", lambda x: f"{x:,.2f}")

## 2.3 Xác định đường dẫn tập dữ liệu mẫu Silver

Sử dụng thư viện `pathlib.Path` để định vị tệp snapshot mẫu `data/silver/rental_latest.parquet` và tệp metadata đồng hành `rental_latest.metadata.json` một cách linh hoạt, hoạt động ổn định bất kể notebook được khởi chạy từ thư mục gốc hay thư mục con `notebooks/`.

In [3]:
PROJECT_ROOT = Path.cwd().resolve()

# Đảm bảo đường dẫn chính xác khi chạy từ thư mục gốc hoặc thư mục con notebooks
if PROJECT_ROOT.name == "notebooks":
    PROJECT_ROOT = PROJECT_ROOT.parent

SILVER_PATH = PROJECT_ROOT / "data" / "silver" / "rental_latest.parquet"
SILVER_METADATA_PATH = PROJECT_ROOT / "data" / "silver" / "rental_latest.metadata.json"

print("Project root          :", PROJECT_ROOT)
print("Silver Parquet path   :", SILVER_PATH)
print("Silver dataset exists :", SILVER_PATH.exists())
print("Silver metadata exists:", SILVER_METADATA_PATH.exists())

Project root          : /mnt/Data/projects/roombeacon
Silver Parquet path   : /mnt/Data/projects/roombeacon/data/silver/rental_latest.parquet
Silver dataset exists : True
Silver metadata exists: True


## 2.4 Nạp dữ liệu vào Pandas DataFrame

Sử dụng phương thức `pd.read_parquet()` để nạp dữ liệu snapshot từ tệp `rental_latest.parquet` vào Pandas DataFrame `df` nhằm kiểm thử môi trường tính toán.

In [4]:
# Nạp snapshot Silver Parquet vào DataFrame
df = pd.read_parquet(SILVER_PATH)

## 2.5 Kiểm tra dữ liệu đã tải

Thực hiện kiểm tra sơ bộ kiểu dữ liệu của biến `df`, kích thước bảng dữ liệu (số dòng, số cột), danh sách các cột đặc trưng và xem trước 5 dòng đầu tiên.

In [5]:
# Kiểm tra kiểu dữ liệu của biến df
type(df)

<class 'pandas.DataFrame'>

Eval Result:
<class 'pandas.DataFrame'>


In [6]:
# Kích thước tập dữ liệu: (số dòng, số cột)
df.shape

(9309, 12)

Eval Result:
(9309, 12)


In [7]:
# Danh sách các cột đặc trưng trong tập dữ liệu
df.columns.tolist()

['source_code', 'rental_post_id', 'source_listing_id', 'title_raw', 'url', 'price_amount', 'area_value', 'location_raw', 'latest_observed_at', 'first_observed_at', 'last_observed_at', 'active_days']

Eval Result:
['source_code', 'rental_post_id', 'source_listing_id', 'title_raw', 'url', 'price_amount', 'area_value', 'location_raw', 'latest_observed_at', 'first_observed_at', 'last_observed_at', 'active_days']


In [8]:
# Xem trước 5 dòng đầu tiên của DataFrame
df.head()

  source_code  rental_post_id         source_listing_id  \
0    nhatrovn            2738  61cc329832cb087a6c8ef6ee   
1    nhatrovn            2736  61ea636e3048d576be907293   
2    nhatrovn            2734  620c98b2f41b6f64646d7f4a   
3    nhatrovn            2693  6319e2561e94754c0b91ef9b   
4    nhatrovn            2707  62b5725aea7aa84b17d4d0ac   

                   title_raw  \
0          CS9 :294/151 XVNT   
1      C2-C3 HOÀNG QUỐC VIỆT   
2               LIÊN KHU 4-5   
3  45/16 ĐƯỜNG 100 BÌNH THỚI   
4              20 ĐƯỜNG SỐ 6   

                                                 url  price_amount  \
0  https://nhatrovn.vn/cho-thue-phong-tro/ho-chi-...  5,200,000.00   
1  https://nhatrovn.vn/cho-thue-phong-tro/ho-chi-...  5,000,000.00   
2  https://nhatrovn.vn/cho-thue-phong-tro/thanh/q...  2,200,000.00   
3  https://nhatrovn.vn/cho-thue-phong-tro/ho-chi-...  1,500,000.00   
4  https://nhatrovn.vn/cho-thue-phong-tro/ho-chi-...  6,500,000.00   

   area_value                  

Eval Result:
  source_code  rental_post_id         source_listing_id  \
0    nhatrovn            2738  61cc329832cb087a6c8ef6ee   
1    nhatrovn            2736  61ea636e3048d576be907293   
2    nhatrovn            2734  620c98b2f41b6f64646d7f4a   
3    nhatrovn            2693  6319e2561e94754c0b91ef9b   
4    nhatrovn            2707  62b5725aea7aa84b17d4d0ac   

                   title_raw  \
0          CS9 :294/151 XVNT   
1      C2-C3 HOÀNG QUỐC VIỆT   
2               LIÊN KHU 4-5   
3  45/16 ĐƯỜNG 100 BÌNH THỚI   
4              20 ĐƯỜNG SỐ 6   

                                                 url  price_amount  \
0  https://nhatrovn.vn/cho-thue-phong-tro/ho-chi-...  5,200,000.00   
1  https://nhatrovn.vn/cho-thue-phong-tro/ho-chi-...  5,000,000.00   
2  https://nhatrovn.vn/cho-thue-phong-tro/thanh/q...  2,200,000.00   
3  https://nhatrovn.vn/cho-thue-phong-tro/ho-chi-...  1,500,000.00   
4  https://nhatrovn.vn/cho-thue-phong-tro/ho-chi-...  6,500,000.00   

   area_value     

## 2.6 Kiểm tra tính duy nhất của tin đăng

Xác thực bất biến danh tính (One-Row-Per-Listing Invariant): Mỗi dòng bắt buộc phải đại diện cho chính xác một bài đăng cho thuê độc nhất (`rental_post_id`).

In [9]:
row_count = len(df)
unique_post_count = df["rental_post_id"].nunique()
duplicate_post_count = df["rental_post_id"].duplicated().sum()

print(f"Tổng số dòng trong DataFrame        : {row_count:,}")
print(f"Số lượng rental_post_id độc nhất    : {unique_post_count:,}")
print(f"Số lượng rental_post_id bị trùng lặp: {duplicate_post_count} (Expected: 0)")

assert duplicate_post_count == 0, "LỖI: Phát hiện trùng lặp rental_post_id!"
print("Xác nhận: Bất biến danh tính được thỏa mãn 100% (Mỗi dòng là duy nhất một tin đăng).")

Tổng số dòng trong DataFrame        : 9,309
Số lượng rental_post_id độc nhất    : 9,309
Số lượng rental_post_id bị trùng lặp: 0 (Expected: 0)
Xác nhận: Bất biến danh tính được thỏa mãn 100% (Mỗi dòng là duy nhất một tin đăng).


## 2.7 Kết quả chuẩn bị dữ liệu và Định hướng tiếp theo

Tổng kết giai đoạn chuẩn bị môi trường:
- Môi trường phân tích dữ liệu với Pandas và Python đã sẵn sàng.
- Snapshot dữ liệu thử nghiệm được nạp trơn tru, xác minh tính toàn vẹn danh tính (`row_count == unique_post_count` và `duplicate == 0`).
- **Định hướng tiếp theo**: Để thực hiện giai đoạn **Initial / Data Quality EDA**, hệ thống sẽ thiết kế và phẳng hóa tập dữ liệu `Raw/Bronze EDA Dataset` bảo toàn các trường thô gốc từ Bronze trước khi tiến hành các phân tích chuyên sâu ở Chương 3.

# Chương 3. Khám phá tổng quan dữ liệu